Analyse de la qualité des donnnées.
##### Objectif :  Evaluer la qualité des données du dataset de marketing
    

Importation des bibliothèques nécessaires et chargement du dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro
from scipy.stats import chi2_contingency

In [ ]:

# Chargement du dataset
df_raw=pd.read_csv("../Data/Raw/marketing_data.csv") #importation du dataset brute sous forme de dataframe
df=df_raw.copy() #copie du dataframe

In [ ]:
#Vérification de la réussite du chargement
df.head()

In [ ]:
# Liste les colonnes 
df.columns

In [ ]:
#Renomination des colonnes du dataset
df.rename(columns={'Education':'Niveau_Education','Year_Birth' : 'Annee_naissance','Marital_Status' : 'Statut_Marital',' Income ' : 'Revenu', 'Kidhome' : 'Nb_Enfants', 'Teenhome' : 'Nb_Ado', 'Dt_Customer': 'Date_Inscription', 'Recency':'Nbjour_Dernier_Achat','MntWines': 'Montant_Vins', 'MntFruits':'Montant_Fruits', 'MntMeatProducts':'Montant_Viandes', 'MntFishProducts':'Montant_Poissons', 'MntSweetProducts':'Montant_Sucreries','MntGoldProds':'Montant_Or', 'NumDealsPurchases':'NbAchat_Remise','NumWebPurchases':'NbAchat_SiteWeb', 'NumCatalogPurchases':'NbAchat_Catalogue','NumStorePurchases':'NbAchat_Magasin','NumWebVisitsMonth':'NbVisit_SiteWebCeMois','AcceptedCmp3':'Campagne3','AcceptedCmp4':'Campagne4', 'AcceptedCmp5':'Campagne5', 'AcceptedCmp2':'Campagne2', 'AcceptedCmp1':'Campagne1', 'Response':'Reponse','Complain':'Plaintes','Country':'Pays'}, inplace=True)


In [ ]:

#Afficher les 5 premières lignes pour vérifier
print(df.head().to_string(index=False))

Analyse et identification des incohérences et des données éronnées 

In [ ]:
#présentation générale  des informations du dataset et du nombre de valeurs non manquantes par colonne
df.shape #utile pour connaitre le nombre de ligne et de colonnes du dataset
df.info()

In [ ]:
#Fouille des colonnes afin d'identifier des anomalies
for val in df.select_dtypes(exclude=["int","datetime","float"]):
    print(f"---------{val}--------")
    print(df[val].unique())
    print(df[val].value_counts())

In [ ]:
#Filtrage de la valeur YOLO 
df.iloc[df["Statut_Marital"]=="YOLO"]

In [ ]:
#Filtrage de la valeur Absurd
df.iloc[df["Statut_Marital"]=="Absurd"]

In [ ]:
# Description statistique des variables catégorielles(moyenne, Q1,Q2,Q3, min, max, nombre de valeurs, ...)
df.describe(include="object")

In [ ]:
# Description statistique des variables numériques (moyenne, Q1,Q2,Q3, min, max, nombre de valeurs, ...)
df.describe(include="int") #ou df.describe()

#df.describe(include="all") #description statistique de tous les types de données

In [ ]:
df.info()

In [ ]:
df["Montant_Total"]=(
    df["Montant_Vins"]
    +df["Montant_Poissons"]
    +df["Montant_Fruits"]
    +df["Montant_Viandes"]
    +df["Montant_Sucreries"]
    +df["Montant_Or"]
)
for val in df[["Revenu","Montant_Total"]]:
    print(f"---------{val}-----------")
    print(df.groupby("Statut_Marital")[val].median())

In [ ]:
for val in df[["Revenu","Montant_Total"]]:
    sns.boxplot(x="Statut_Marital", y=val, data=df)
    plt.show()

Observation :
###### Nous remarquons que Date_Inscription est de type object ce qui n'est pas correcte. Nous allons donc le transformer en type de date.
###### Nous remarquons les colonnes portant sur les campagnes et réponses sont de type int. Nous allons donc les transformer en booléen
###### Nous manipulons mieux les âges que les années. Nous allons insérer la colonne de Age. Il sera calculé en tenant compte de 2014 comme année de référence car  dans le dataset, les activités se sont déroulées de 2012 à 2014,aussi les colonnes de la date d'inscription et du nombre jour effectué après le dernier acaht, nous font comprendre que les clients sont toujours actifs en 2014
###### Dans la colonne Niveau_Education nous avons remarqué Graduation qui n'est pas un niveau d'éducation. Nous devons donc le remplacer par Graduate
###### Dans la colonne Statut_Marital, nous avons d'un côté YOLO et Abusurd qui ne sont pas des statuts matrimoniaux et seront supprimés car ils sont très peu
###### De l'autre Alone et Single seront fusionnés car ils ont le même sens
###### Aussi, Married et Together seront fusionnés. En effet, nous avons étudié la similarité de Married et Together tant statistiquement(mediane par groupe des Revenu et Montant_Total) que graphiquement(boxplot)


Traitement des incohérences et données erronées

In [ ]:

df['Date_Enrollement']=pd.to_datetime(df['Date_Inscription'])
df.drop('Date_Inscription', axis='columns', inplace=True) #suppression de la colonne Date_inscription

In [ ]:
#Conversion de int en bool des colonnes campagnes, reponses, plaintes
campagne_type=["Campagne1","Campagne2","Campagne3","Campagne4","Campagne5","Reponse", "Plaintes"]
for val in campagne_type:
    df[val]=df[val].astype(bool)

In [ ]:
#Trouvons l'âge des clients. Nous allons le faire en tenant compte de 2014
# Car les activités se sont déroulés de 2012 à 2014
from datetime import datetime
df['Age']=2014-df['Annee_naissance']

In [ ]:
#Remplacement de Gradution par Graduate 
df["Niveau_Education"]=df["Niveau_Education"].replace({"Graduation":"Graduate"}, inplace=True) 

In [ ]:
#Fusion de Single et Alone
df["Statut_Marital"]=df["Statut_Marital"].replace("Alone","Single")

#Fusion de Married et Together
df["Statut_Marital"]=df["Statut_Marital"].replace("Together","Married")

#Suppression des valeurs YOLO et Together
df=df[~df["Statut_Marital"].isin(["YOLO","Absurd"])]

In [ ]:
df["Age"]

In [ ]:
df["Statut_Marital"].unique()

Analyse et identifaction des doublons

In [ ]:
#nombre de lignes exactement identiques
df.duplicated().sum()

Observation: Il n'y a aucun doublon

Analyse et indentification des valeurs manquantes

In [ ]:
#comptage des valeurs manquantes par colonne
df.isnull().sum()

In [ ]:
df[df["Revenu"].isna()]

###### Nous remarquons que ces clients qui ont des revenus manquants,ont des dépenses enregistrées concernant les fruits, poissons et autres. Les valeurs manquantes ne signifient pas ces clients n'ont pas de revenu

In [ ]:
#Nous devons avant tout trouver le type de valeurs manquantes(MCAR, MAR,MNAR) avant tout traitement
#Nous suspectons les revenus manquants de dépendre du niveau d'éducation et du statu matrimonial, du nombre d'enfants et de l'âge
#Procédons au test de contingence(d'indépendance) du khi-deux
#Evalution de la relation entre les valeurs manquantes et les colonnes suspectées
#Création de la table de contingence
def test_contingence(df,colonne):
    df["Revenu_Manquant"]=df["Revenu"].isna()
    table=pd.crosstab(df[colonne],df["Revenu_Manquant"])
    chi2, p, dof, expected=chi2_contingency(table) #chi2: Statistique de test; p:p-value; dof:degré de liberté; expected : Effectif théorique
    print(f"----------{colonne}------------")
    print(table)
    print(f"Statistique de khi-2:{chi2}")
    print(f"p-value: {p}")
    if p>0.05:
        print(f"Pas de relation significative avec les valeurs manquantes")
    else:
        print(f"Relattion significative avec les valeurs manquantes")

In [ ]:
# Transformons l'âge en groupe d'age (catégorie)
df["Groupe_Age"]=pd.cut(df["Age"],bins=[18,25,35,45,60,100],
labels=["18-25","25-35","35-45","45-60","60+"])

In [ ]:
#Evaluation de la dépendance entre les valeurs manquantes et les variables suspectées
var=["Groupe_Age","Niveau_Education","Statut_Marital","Nb_Enfants","Nb_Ado"]
for i in var:
    test_contingence(df,i)

In [ ]:
#Calcul des proportions de revenus manquants par nombre d'enfants
Revenu_Manquant=df["Revenu"].isna()
Prop_Manquant=df.groupby("Nb_Enfants")["Revenu_Manquant"].mean()*100
Prop_Manquant

Observation :
###### Nous suspections les valeurs manquantes de dépendre de certaines variables comme l'âge, le niveau d'éducation, le statut matrimonial, le nombre d'enfants et d'ado.
###### Un test de d'indépendance de chi-deux a été réalisé pour dissiper les zones d'ombre. Nous avons remarqué que seul le nombre d'enfants peut expliquer les revenus manquants avec une p-value égale à 0,049 inferieure au seuil requis(0,05). Les données manquantes sont donc de type MAR(Missed At Random)
###### Aussi, nous avons remarqué que les clients qui ont beaucoup d'enfants ont une proportion de revenu manquants élevée.

Imputation et traitement des valeurs manquantes
###### Nous devons choisir entre l'imputation par la moyenne et celle par la médiane. Il faut donc étudier la normalité de la distribution de la série de données afin de faire le choix

In [ ]:
#Vérifions la nature de la distribution de la série de données(Revenu)
#Vérification de la normalisattion des données

def test_shapiro(df):
    for i in df.select_dtypes(include='number').columns:
        stat_test, p=shapiro(df[i])
        print(i)
        if p>0.05 :
            print("Les données suivent la loi normale")
        else:
            print("Les données ne suivent par la loi normale")  
test_shapiro(df)

Observation:
###### Les données de la colonne Revenu ne suivent pas la loi normale. Nous optons donc pour l'imputation par la médiane. Nous allons faire une imputation conditionnelle. C'est-à-dire, en tenant compte des différents groupes des revenus des clients par Nb_Enfants nous allons remplacer chaque valeur manquante par la médiane de son groupe correspondant

In [ ]:
#Imputation conditionnelle
df["Revenu"]=df["Revenu"].fillna(df.groupby("Nb_Enfants")["Revenu"].transform("median"))

Conclusion : Les valeurs manquantes sont  traitées

Analyse et identification des valeurs aberrantes

###### Nous connaissons déjà que les donnéees ne suivent paar la loi normale. Dans ce cas, nous allons procéder par une analyse graphique et  une analyse statistique basée sur la méthode de l'IQR pour identifier et analyser les valeurs aberrantes. Si les données suivaient la loi normale, nous aurions une analyse statistique basée sur la méthode du Z-score

In [ ]:
#Analyse graphique : Cherchons les outliers
#Fusionnons les nombres d'achats
df["NbAchat_Total"]=(
    df["NbAchat_Catalogue"]
    +df["NbAchat_Remise"]
    +df["NbAchat_SiteWeb"]
    +df["NbAchat_Magasin"]
)
for val in df[["Revenu","Montant_Total","Age", "NbAchat_Total", "Nbjour_Dernier_Achat"]]:
    df.boxplot(val)
    plt.show()

Note : 
###### Regrouper en Montant_Total et NbAchat_Total permet d'éviter de considérer comme aberrant une préférence de produit ou de canal d'achat

In [ ]:
#Analyse statistique: Détection des valeurs aberrantes avec l'approche de l'IQR
def valeur_aberrante(df,colonne):
    if colonne in df.select_dtypes(include="number").columns:
        valeur=df[colonne].sort_values(ascending=True)
        q1=valeur.quantile(0.25)        #ou q1=np.percentile(valeur, 25)
        q2=valeur.quantile(0.50)        #ou q2=np.percentile(valeur,50)
        q3=valeur.quantile(0.75)        #ou q3=np.percentile(valeur,75)
        iqr=q3-q1
        born_inf=q1-1.5*iqr
        born_sup=q3+1.5*iqr
        aberrantes=valeur[(valeur < born_inf) | (valeur> born_sup) ]
        print(f"----- {colonne} -----")
        print(f"Minimum:{df[colonne].min()}")
        print(f"Maximum:{df[colonne].max()}")
        print(f"Q1 = {q1}, Q2 = {q2}, Q3 = {q3}")
        print(f"Borne inférieure = {born_inf}, Borne supérieure = {born_sup}")
        if aberrantes.empty:
            print("Aucune valeur aberrante détectée.")
        else:
            print(f"le nombre de valeurs aberrantes est :{aberrantes.count()}\n")
            print("Valeurs aberrantes détectées :")
            print(aberrantes.to_string(index=False))
        print("\n")
    else :
        print(f"Les valeurs aberrantes n'existent pas pour des variables catégorielles")

In [ ]:
#Appel de la fonction en considérant les variables suspectées
for val in df[["Revenu","Montant_Total","Age", "NbAchat_Total", "Nbjour_Dernier_Achat"]]:
    valeur_aberrante(df,val)

In [ ]:
df.iloc[df["Revenu"]>150000]

In [ ]:
df.iloc[df["Montant_Total"]>2000]

In [ ]:
df.iloc[df["NbAchat_Total"]>40]

In [ ]:
df.iloc[df["Age"]>100]

In [ ]:
#Vérification de la relation entre Revenu et Montant_Total
df[["Revenu","Montant_Total"]].corr() #il existe une relation positive mais pas entre ces variables qui ne sont pas parfaites

In [ ]:
#Suppression de la valeur 666666
df=df[~(df["Revenu"]>250000)]

#Suppression de l'âge
df=df[~(df["Age"]>100)]

Conclusion:
###### Il sied de souligner que toutes les valeurs aberrantes sont pas des erreurs. C'est le cas de des colonnes Montant_Total et NbAchat_Total qui présentent des outliers très proches de la borne supérieure. Elles peuvent être expliquer par le fait que certains clients ont un pouvoir d'achat élevé

###### L'analyse tant statistque que graphique de la colonne Revenu nous permis de constater des valeurs aberrantes assez proches de la borne supérieure et une très éloignéé(extrême) qui est 666666. Nous l'avons donc supprimé car  sa proportion est très faible. 
###### Aussi, d'autres valeurs aberrantes ont été acceptées car elles se rapprochent de la tendance des des revenus des clients

###### L'analyse tant statistique que graphique de la colonne Age nous permis de connstater trois valeurs aberrantes qui sont 114, 115 et 121ans. Ces valeurs sont loin d'être plausibles et tenant compte de leur faible proportion sur l'âge des clients nous les avons supprimé

Fin de la phase de nettoyage et export du dataset en fichier csv

In [ ]:
#Export du dataset en fichier csv
df.to_csv("C:/Users/Kardo BALOSSA/Data_Analyis_Projects/marketing-data-analyst-project/Data/Clean", index=False)